# Latent diffusion pretraining: metrics and inference

The notebook inspects a `train_ldm.py` run: plots from `metrics.csv`, the sample grids saved during training,
and generation from EMA weights with a breakdown by denoising step, in latent space and after the VAE decoder.

Cells can be rerun while training is going: they reread the files from the run folder. Generation needs a GPU.

In [ ]:
import csv
import io
import json
import math
from pathlib import Path

import cv2
import ipywidgets as widgets
import matplotlib.pyplot as plt
import numpy as np
import torch
from diffusers import AutoencoderKL
from IPython.display import display
from matplotlib import colormaps, font_manager
from PIL import Image, ImageDraw, ImageFont

from models import UNet
from train import VPSDE

RUN_DIR = Path('runs/ldm_v3')   # ldm_v1: first run, diverged at step 5400
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

cfg = json.loads((RUN_DIR / 'config.json').read_text())
print(f"steps {cfg['steps']}, batch {cfg['batch']} x {cfg['devices']} GPU, pred {cfg['pred']}, base {cfg['base']}, "
      f"head {cfg.get('head_ch') or 'single'}, qk-norm {cfg.get('qk_norm', False)}, "
      f"lr {cfg['lr']} -> {cfg['lr_min']}, warmup {cfg.get('warmup', 0)}, VAE {cfg['vae']}, scale {cfg['scale']:.5f}")

## Metrics

The loss here is unweighted eps-MSE, as in `train.py`: for `v` it is converted to noise error, so it is comparable with earlier runs.
Intervals of t: at small t the model fills in fine details, at large t it guesses the overall composition.
The dashed line marks a restart from the last checkpoint.

The bottom row warns about divergence earlier than the loss: gradient norm before clipping (clipping at 1) and the worst attention block.
Attention entropy as a fraction of log N: near 1 the attention is spread over the whole map, near 0 it looks at a single point.
In ldm_v1, before the divergence the entropy at 16x16 was 0.01 with logits up to 18 000.

In [ ]:
def read_metrics(path):
    """metrics.csv -> {column: np.array} and the steps from which training resumed after a stop.
    After a restart the steps since the last checkpoint are written again, so the last record of each step is taken."""
    with open(path) as f:
        rows = list(csv.DictReader(f))
    restarts = [int(r['step']) for prev, r in zip(rows, rows[1:]) if int(r['step']) <= int(prev['step'])]
    by_step = {int(r['step']): r for r in rows}
    rows = [by_step[s] for s in sorted(by_step)]
    return {k: np.array([float(r[k]) for r in rows]) for k in rows[0]}, restarts


def smooth(y, k=0.9):
    """Exponential smoothing, as in TensorBoard."""
    out, acc = np.empty_like(y), y[0]
    for i, v in enumerate(y):
        acc = k * acc + (1 - k) * v
        out[i] = acc
    return out


log, restarts = read_metrics(RUN_DIR / 'metrics.csv')
s = log['step']
bins = ['loss_t0-0.2', 'loss_t0.2-0.4', 'loss_t0.4-0.6', 'loss_t0.6-0.8', 'loss_t0.8-1']

extra = 'grad_norm' in log                     # ldm_v1 did not have these columns yet
fig, axs = plt.subplots(3 if extra else 2, 2, figsize=(16, 15 if extra else 10))
for ax in axs.flat:
    ax.grid(alpha=0.5, zorder=1, which='both')
    ax.set_xlabel('step')
    for r in restarts:
        ax.axvline(r, color='gray', ls=':', zorder=1)

# 1. overall loss
axs[0, 0].semilogy(s, log['loss'], color='C0', alpha=0.3, zorder=2)
axs[0, 0].semilogy(s, smooth(log['loss']), color='C0', zorder=2)
axs[0, 0].set_title('eps-MSE, mean over t (faint line unsmoothed)')

# 2. loss by t interval
for i, b in enumerate(bins):
    axs[0, 1].semilogy(s, smooth(log[b]), color=f'C{i}', label='t ' + b.replace('loss_t', ''), zorder=2)
axs[0, 1].set_title('eps-MSE by t interval, smoothed')
axs[0, 1].legend()

# 3. learning rate
axs[1, 0].semilogy(s, log['lr'], zorder=2)
axs[1, 0].set_title('learning rate')

# 4. speed
axs[1, 1].plot(s, log['it_per_s'], zorder=2)
axs[1, 1].set_title('steps per second')
axs[1, 1].set_ylim(0, None)

if extra:
    # 5. gradient norm before clipping: mean and max over the log interval
    axs[2, 0].semilogy(s, log['grad_norm'], label='mean', zorder=2)
    axs[2, 0].semilogy(s, log['grad_norm_max'], label='max', alpha=0.6, zorder=2)
    axs[2, 0].axhline(cfg['grad_clip'], color='k', ls='--', label='clipping threshold', zorder=2)
    axs[2, 0].set_title('gradient norm before clipping')
    axs[2, 0].legend()

    # 6. worst attention block
    ax = axs[2, 1]
    ax.plot(s, log['attn_min_entropy'], color='C3', zorder=2)
    ax.set_ylabel('minimum entropy', color='C3')
    ax.set_ylim(0, 1)
    ax2 = ax.twinx()
    ax2.semilogy(s, log['attn_max_logit'], color='C0', zorder=2)
    ax2.set_ylabel('maximum logit', color='C0')
    ax.set_title('worst attention block')

plt.tight_layout()
plt.show()

left = (cfg['steps'] - s[-1]) / max(log['it_per_s'][-1], 1e-9) / 3600
print(f"last step {int(s[-1])} of {cfg['steps']}, loss {smooth(log['loss'])[-1]:.4f}, ~{left:.1f} h left")

In [ ]:
# entropy and maximum logit of each attention block (attn.csv, available since ldm_v2)
if (RUN_DIR / 'attn.csv').exists():
    att, _ = read_metrics(RUN_DIR / 'attn.csv')
    layers = [k[:-4] for k in att if k.endswith('_ent')]
    fig, axs = plt.subplots(1, 2, figsize=(16, 5.5))
    for i, n in enumerate(layers):
        c = plt.cm.viridis(i / max(len(layers) - 1, 1))
        axs[0].plot(att['step'], att[n + '_ent'], color=c, label=n, zorder=2)
        axs[1].semilogy(att['step'], att[n + '_max'], color=c, label=n, zorder=2)
    axs[0].set_title('attention entropy by block, fraction of log N')
    axs[0].set_ylim(0, 1)
    axs[1].set_title('maximum logit by block')
    for ax in axs:
        ax.grid(alpha=0.5, zorder=1, which='both')
        ax.set_xlabel('step')
    axs[1].legend(fontsize=8, ncol=2)
    plt.tight_layout()
    plt.show()
else:
    print('no attn.csv: attention was not logged in this run')

## Samples saved during training

Every 2500 steps: 16 images from the same noise, EMA weights. In ldm_v2 this is Heun 50 (smeared), from ldm_v3 on SDE 250.
At the end of training the grid `final_sde1000` is added: the same 16 noises, SDE with 1000 steps.

In [ ]:
WIDTH = 768    # grid side on screen, in pixels

grids = sorted((RUN_DIR / 'samples').glob('step_*.png')) + sorted((RUN_DIR / 'samples').glob('final_*.png'))
if grids:
    pngs = [p.read_bytes() for p in grids]
    img = widgets.Image(value=pngs[-1], format='png', layout=widgets.Layout(width=f'{WIDTH}px', height=f'{WIDTH}px'))
    info = widgets.HTML()
    play = widgets.Play(min=0, max=len(grids) - 1, value=len(grids) - 1, interval=800, description='play')
    slider = widgets.IntSlider(min=0, max=len(grids) - 1, value=len(grids) - 1, description='grid',
                               layout=widgets.Layout(width=f'{WIDTH - 80}px'))
    widgets.jslink((play, 'value'), (slider, 'value'))

    def show_grid(i):
        img.value = pngs[i]
        info.value = f'<b>{grids[i].stem}</b>, {i + 1} of {len(grids)}'

    slider.observe(lambda change: show_grid(change['new']), names='value')
    show_grid(len(grids) - 1)
    display(widgets.VBox([widgets.HBox([play, slider]), info, img]))
else:
    print('no samples yet, the first appear at step 2500')

In [ ]:
# first row of each grid: the same 4 noises at every training step
ROW = 2 + 512 + 2   # padding + 512x512 image + padding

rows = [np.asarray(Image.open(p).reduce(2))[:ROW // 2] for p in grids]
if rows:
    fig, ax = plt.subplots(figsize=(14, 3.5 * len(rows)))
    ax.imshow(np.concatenate(rows, axis=0))
    ax.set_yticks([ROW // 2 * (i + 0.5) for i in range(len(rows))])
    ax.set_yticklabels([p.stem.replace('step_', '') for p in grids])
    ax.set_xticks([])
    ax.set_title('row = training step')
    plt.show()

---
## Model

EMA weights are taken from `ema_*.pt` (every 5000 steps) or from `ckpt/last.ckpt` (the last Lightning checkpoint, including the final one).
Model latents have variance 1; before the decoder they are divided by `scale`, as in training.

In [ ]:
vae = AutoencoderKL.from_pretrained(cfg['vae'], local_files_only=True, low_cpu_mem_usage=False)
vae = vae.to(DEVICE).eval().requires_grad_(False)
SCALE = cfg['scale']
sde = VPSDE(pred=cfg['pred'])
CMAP = colormaps['RdBu_r']


def load_ema(path):
    """EMA weights of the U-Net from ema_*.pt or from Lightning's ckpt/last.ckpt. Returns the model and the training step."""
    ck = torch.load(path, map_location='cpu', weights_only=False, mmap=True)
    if 'ema' in ck:
        sd, step = ck['ema'], ck['step']
    else:
        sd, step = {k[4:]: v for k, v in ck['state_dict'].items() if k.startswith('ema.')}, ck['global_step']
    model = UNet(img_size=64, in_ch=4, base=cfg['base'], mults=[int(v) for v in cfg['mults'].split(',')],
                 num_res=cfg['num_res'], attn_res=[int(v) for v in cfg['attn_res'].split(',')],
                 dropout=cfg['dropout'], norm_groups=32,
                 head_ch=cfg.get('head_ch') or None, qk_norm=cfg.get('qk_norm', False))   # ldm_v1 did not have them yet
    model.load_state_dict(sd)
    return model.to(DEVICE).eval().requires_grad_(False), step


@torch.no_grad()
def decode(z, chunk=8):
    """Model latents (B, 4, 64, 64) -> uint8 images (B, 3, 512, 512) on CPU."""
    out = []
    for i in range(0, len(z), chunk):
        x = vae.decode(z[i:i + chunk].to(DEVICE).float() / SCALE).sample
        out.append(((x.clamp(-1, 1) + 1) * 127.5).round().byte().cpu())
    return torch.cat(out)


def to_np(img):
    """(3, H, W) uint8 -> (H, W, 3) for imshow."""
    return img.permute(1, 2, 0).numpy()


def latent_tile(z, lim=3.0):
    """(4, 64, 64) -> (128, 128, 3) in [0, 1]: the four latent channels as a 2x2 square,
    blue below zero, red above, saturating at +-lim."""
    z = z.float()
    m = torch.cat([torch.cat([z[0], z[1]], 1), torch.cat([z[2], z[3]], 1)], 0).numpy()
    return CMAP((m / lim + 1) / 2)[..., :3]


@torch.no_grad()
def generate(model, n, seed=0, method='sde', steps=250, rho=None, n_keep=None, guide=None, w=1.0):
    """
    The same steps as sample() in train.py, but with history.
      method : 'sde' (Euler-Maruyama), 'heun' (ODE, two network calls per step) or 'euler' (ODE)
      rho    : grid concentration in t toward zero, default 2 for ODE and 1 for SDE, as in training
      n_keep : how many steps to keep in the history, evenly spaced, default all
      guide, w : autoguidance (Karras et al. 2024). guide is a weak model, e.g. an early EMA snapshot,
               output = guide + w (model - guide), w = 1 means no guidance
    Returns the final latents (n, 4, 64, 64) on CPU and history: a list of dicts with step index i, t, sigma,
    x (current noisy latent x_t) and x0 (the model's guess of the final latent), in float16 on CPU.
    """
    rho = rho if rho is not None else (1.0 if method == 'sde' else 2.0)
    g = torch.Generator(device=DEVICE).manual_seed(seed)
    x = torch.randn(n, 4, 64, 64, generator=g, device=DEVICE)
    ts = sde.t_min + (1.0 - sde.t_min) * np.linspace(1.0, 0.0, steps + 1) ** rho
    keep = set(np.linspace(0, steps, min(n_keep or steps + 1, steps + 1)).round().astype(int).tolist())

    def eps_at(x, t):
        tt = torch.full((n,), float(t), device=DEVICE)
        with torch.autocast('cuda', dtype=torch.bfloat16):
            e = model(x, sde.c(tt)).float()
            if guide is not None:
                g = guide(x, sde.c(tt)).float()
                e = g + w * (e - g)
        return sde.to_eps(e, x, tt)

    def drift(x, t, e, k=1.0):
        # right-hand side when moving backward: x <- x + h * drift;  k = 1 for ODE, k = 2 for SDE
        tt = torch.tensor(float(t), device=DEVICE)
        return 0.5 * sde.beta(tt) * (x - k * e / sde.sigma(tt))

    history = []
    for i in range(steps + 1):
        t = ts[i]
        tt = torch.tensor(float(t), device=DEVICE)
        e = eps_at(x, t)
        x0 = (x - sde.sigma(tt) * e) / sde.m(tt)
        if i in keep:
            history.append(dict(i=i, t=float(t), sigma=float(sde.sigma(tt)), x=x.half().cpu(), x0=x0.half().cpu()))
        if i == steps:
            break
        h = t - ts[i + 1]
        if method == 'heun':
            d = drift(x, t, e)
            x_e = x + h * d
            x = x + 0.5 * h * (d + drift(x_e, ts[i + 1], eps_at(x_e, ts[i + 1])))
        elif method == 'euler':
            x = x + h * drift(x, t, e)
        elif method == 'sde':
            x = x + h * drift(x, t, e, k=2.0)
            if i < steps - 1:
                x = x + math.sqrt(float(sde.beta(float(t))) * h) * torch.randn(x.shape, generator=g, device=DEVICE)
        else:
            raise ValueError(method)
    return x0.float().cpu(), history

## Generation

The sampler strongly affects the result (comparison in `runs/ldm_v2/compare/`, `tools/compare_samplers.py`):
Heun (ODE) gives smeared images for any grid and number of steps, SDE at 50-100 steps leaves noise
that the VAE decoder turns into crumpled foil. SDE with 250-1000 steps gives the best result, autoguidance with the early
snapshot 5000 and w 1.5-2 makes faces and bodies noticeably more coherent.

In [ ]:
# ------------------------------------------------------------- parameters
CKPT = 'ckpt/last.ckpt'          # or 'ema_0025000.pt'
GUIDE = Path('runs/ldm_v4/ckpt/last.ckpt')   # weak model for autoguidance, None = without it.
# For ldm_v3 the best guide is the final ldm_v2: the same model but less trained. Early snapshots of ldm_v3 itself are barely weaker than the final one
W = 2.0                          # autoguidance strength: 1 = none. ldm_v3 with guide ldm_v2: 2-3, with ema_0005000 from ldm_v2: 1.25-1.5
SEED = 2
BATCH = 16
METHOD = 'sde'                   # 'sde', 'heun' or 'euler'. Heun gives smeared images, SDE 50 leaves noise
STEPS = 250                      # SDE: 250 is enough, 1000 is slightly better
N_KEEP = 51                      # steps in the history for the breakdown below
COLS = 4

# ------------------------------------------------------------- generation
model, step = load_ema(RUN_DIR / CKPT)
guide, guide_step = load_ema(GUIDE) if GUIDE else (None, None)
final, history = generate(model, BATCH, seed=SEED, method=METHOD, steps=STEPS, n_keep=N_KEEP, guide=guide, w=W)
imgs = decode(final)
del guide
torch.cuda.empty_cache()

# each image in a separate file, 512x512
# guide in the name: its run and step, runs/ldm_v2/ckpt/last.ckpt -> ag_ldm_v2-26000
guide_run = (Path(GUIDE).parent.parent if Path(GUIDE).parent.name == 'ckpt' else Path(GUIDE).parent).name if GUIDE else ''
name = f'step{step:07d}_{METHOD}{STEPS}_seed{SEED}' + (f'_ag_{guide_run}-{guide_step}_w{W}' if GUIDE else '')
out_dir = RUN_DIR / 'generated' / name
out_dir.mkdir(parents=True, exist_ok=True)
for i, im in enumerate(imgs):
    Image.fromarray(to_np(im)).save(out_dir / f'{i:03d}.png')
print(f'training step {step}, {BATCH} images saved to {out_dir}')

# ------------------------------------------------------------- figure
rows = math.ceil(BATCH / COLS)
fig, axs = plt.subplots(rows, COLS, figsize=(4.5 * COLS, 4.7 * rows), squeeze=False)
for ax in axs.flat:
    ax.axis('off')
for i, im in enumerate(imgs):
    axs.flat[i].imshow(to_np(im))
    axs.flat[i].set_title(str(i))
plt.suptitle(f'{name}', fontsize=15)
plt.tight_layout()
plt.show()

## Breakdown by denoising step

Four rows for one image:

1. `x_t` in latent space: four channels as a 2x2 square, blue below zero, red above.
2. `x_t` through the VAE decoder. The decoder is not trained on noisy latents, so at the start this is not an image but its reaction to noise.
3. `x̂_0` in latent space: what the model at this step takes to be the final latent, `x̂_0 = (x_t - σ ε̂) / m`.
4. `x̂_0` through the decoder: what this guess looks like.

The next cell decodes all history steps for the chosen image; the slider below works on them.

In [ ]:
IDX = 15          # image index in the batch, it is labeled above the image in the figure above
N_FRAMES = 8     # frames in the figure

dec_x = decode(torch.stack([fr['x'][IDX] for fr in history]))
dec_x0 = decode(torch.stack([fr['x0'][IDX] for fr in history]))

pick = np.linspace(0, len(history) - 1, N_FRAMES).round().astype(int)
fig, axs = plt.subplots(4, N_FRAMES, figsize=(2.6 * N_FRAMES, 11.5))
for ax in axs.flat:
    ax.set_xticks([])
    ax.set_yticks([])
for c, j in enumerate(pick):
    fr = history[j]
    axs[0, c].imshow(latent_tile(fr['x'][IDX]), interpolation='nearest')
    axs[0, c].set_title(f"step {fr['i']}, t = {fr['t']:.3f}\n$\\sigma$ = {fr['sigma']:.3f}", fontsize=10)
    axs[1, c].imshow(to_np(dec_x[j]))
    axs[2, c].imshow(latent_tile(fr['x0'][IDX]), interpolation='nearest')
    axs[3, c].imshow(to_np(dec_x0[j]))
for r, label in enumerate(['$x_t$, latent', '$x_t$, VAE', '$\\hat x_0$, latent', '$\\hat x_0$, VAE']):
    axs[r, 0].set_ylabel(label, fontsize=12)
plt.suptitle(f'image {IDX}, {name}', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
SIDE = 320    # side of one panel on screen, in pixels


def png(arr):
    """(H, W, 3) uint8 or float in [0, 1] -> png bytes for widgets.Image."""
    if arr.dtype != np.uint8:
        arr = (arr * 255).round().astype(np.uint8)
    buf = io.BytesIO()
    Image.fromarray(arr).save(buf, 'png')
    return buf.getvalue()


# enlarge the latent without smoothing so the individual 64x64 cells are visible
big = lambda a: np.kron(a, np.ones((4, 4, 1)))
panels = [[png(big(latent_tile(fr['x'][IDX]))) for fr in history],
          [png(to_np(im)) for im in dec_x],
          [png(big(latent_tile(fr['x0'][IDX]))) for fr in history],
          [png(to_np(im)) for im in dec_x0]]
titles = ['x_t, latent', 'x_t, VAE', 'x̂_0, latent', 'x̂_0, VAE']
last = len(history) - 1

box = widgets.Layout(width=f'{SIDE}px', height=f'{SIDE}px')
imgs_w = [widgets.Image(value=p[0], format='png', layout=box) for p in panels]
cols = [widgets.VBox([widgets.HTML(f'<b>{t}</b>'), w]) for t, w in zip(titles, imgs_w)]
info = widgets.HTML()

# Play moves the slider by itself: the whole history in about 10 s, no faster than 50 ms per frame
play = widgets.Play(min=0, max=last, interval=max(50, 10_000 // len(history)), description='play')
slider = widgets.IntSlider(min=0, max=last, description='step', layout=widgets.Layout(width=f'{4 * SIDE - 80}px'))
widgets.jslink((play, 'value'), (slider, 'value'))


def show_frame(k):
    for w, p in zip(imgs_w, panels):
        w.value = p[k]
    fr = history[k]
    info.value = (f"<b>image {IDX}, {name}</b>: step {fr['i']} of {STEPS}, t = {fr['t']:.3f}, "
                  f"&sigma; = {fr['sigma']:.3f}, std x_t = {fr['x'][IDX].float().std():.2f}")


slider.observe(lambda change: show_frame(change['new']), names='value')
show_frame(0)
display(widgets.VBox([widgets.HBox([play, slider]), info, widgets.HBox(cols)]))

# ------------------------------------------------------------- the same animation to files
VIDEO_SECONDS = 10   # pass through the whole history, the last frame is held for another 2 s
GIF_SIDE = 256       # panel side in the gif: gif barely compresses, so it is smaller than the mp4
FONT = ImageFont.truetype(font_manager.findfont('DejaVu Sans'), 26)   # the default PIL font has no Cyrillic


def video_frame(k, side=512):
    """Frame k: caption on top and four panels in a row, as in the widget. (H, W, 3) uint8."""
    fr = history[k]
    tiles = [big(latent_tile(fr['x'][IDX])), to_np(dec_x[k]), big(latent_tile(fr['x0'][IDX])), to_np(dec_x0[k])]
    head = 84
    canvas = Image.new('RGB', (4 * side, side + head), (255, 255, 255))
    d = ImageDraw.Draw(canvas)
    d.text((10, 8), f"image {IDX}, {name}: step {fr['i']} of {STEPS}, t = {fr['t']:.3f}, σ = {fr['sigma']:.3f}",
           fill=(0, 0, 0), font=FONT)
    for j, (title, a) in enumerate(zip(titles, tiles)):
        a = a if a.dtype == np.uint8 else (a * 255).round().astype(np.uint8)
        canvas.paste(Image.fromarray(a).resize((side, side), Image.NEAREST), (j * side, head))
        d.text((j * side + 10, 46), title, fill=(0, 0, 0), font=FONT)
    return np.asarray(canvas)


frames = [video_frame(k) for k in range(len(history))]
fps = max(len(history) / VIDEO_SECONDS, 1)
stem = out_dir / f'trajectory_{IDX:03d}'
H, W = frames[0].shape[:2]

# in this build OpenCV only has mp4v (no H.264): QuickTime and VLC open it, browsers not always
vw = cv2.VideoWriter(str(stem.with_suffix('.mp4')), cv2.VideoWriter_fourcc(*'mp4v'), fps, (W, H))
for f in frames + [frames[-1]] * round(2 * fps):
    vw.write(np.ascontiguousarray(f[..., ::-1]))   # OpenCV expects BGR
vw.release()

small = [Image.fromarray(f).resize((4 * GIF_SIDE, round(H * 4 * GIF_SIDE / W)), Image.LANCZOS) for f in frames]
small[0].save(stem.with_suffix('.gif'), save_all=True, append_images=small[1:], loop=0,
              duration=[round(1000 / fps)] * (len(small) - 1) + [2000])
print(f"video: {stem.with_suffix('.mp4')} ({W}x{H}, {len(frames)} frames, {fps:.1f} fps), gif: {stem.with_suffix('.gif')}")

---
## How the model changed during training

The same noises through every saved checkpoint, SDE with 250 steps. Row = training step.

In [ ]:
N_CMP = 6
SEED_CMP = 0

ckpts = sorted(RUN_DIR.glob('ema_*.pt')) + [RUN_DIR / 'ckpt' / 'last.ckpt']
results = []
for p in ckpts:
    m, st = load_ema(p)
    fin, _ = generate(m, N_CMP, seed=SEED_CMP, method='sde', steps=250, n_keep=1)
    results.append((st, decode(fin)))
    del m
    torch.cuda.empty_cache()

fig, axs = plt.subplots(len(results), N_CMP, figsize=(2.8 * N_CMP, 2.9 * len(results)), squeeze=False)
for r, (st, ims) in enumerate(results):
    for c in range(N_CMP):
        axs[r, c].imshow(to_np(ims[c]))
        axs[r, c].set_xticks([])
        axs[r, c].set_yticks([])
    axs[r, 0].set_ylabel(f'step {st}', fontsize=12)
plt.tight_layout()
plt.show()